In [5]:
import pandas as pd
from pathlib import Path
from IPython.display import display

DATASET_PATH = Path("old_sensor_datasets")

people = ["Hiruni", "Mineth", "Tanushka", "Thinula"]

activities = ["Bending", "Idle", "Picking", "Pushing"]

activity_labels = {
    "Bending": 0,
    "Idle": 1,
    "Picking": 2,
    "Pushing": 3
}

merged_out_dir = Path("old_sensor_merged")
merged_out_dir.mkdir(parents=True, exist_ok=True)


def read_sensor_file(file_path: Path) -> pd.DataFrame:
    df = pd.read_csv(file_path, skiprows=11)

    required_columns = ["FreeAcc_X", "FreeAcc_Y", "FreeAcc_Z"]

    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise KeyError(f"Missing columns in {file_path}: {missing_columns}")

    acc_df = df[required_columns].copy()

    acc_df.rename(columns={
        "FreeAcc_X": "Acc_X",
        "FreeAcc_Y": "Acc_Y",
        "FreeAcc_Z": "Acc_Z"
    }, inplace=True)

    return acc_df


summary_rows = []

for person in people:
    person_merged_rows = []

    person_folder = DATASET_PATH / person

    if not person_folder.exists():
        summary_rows.append({
            "Person": person,
            "Activity": "ALL",
            "Folders Found": 0,
            "Folders Processed": 0,
            "Rows Added": 0,
            "Status": "Person folder not found"
        })
        continue

    for activity in activities:
        activity_label = activity_labels[activity]

        activity_folders = sorted([
            folder for folder in person_folder.glob(f"{person}_{activity}*")
            if folder.is_dir()
        ])

        folders_processed = 0
        rows_added = 0
        status = "OK"

        for activity_folder in activity_folders:
            try:
                left_file = sorted(activity_folder.glob("2B_Left*.csv"))[0]
                right_file = sorted(activity_folder.glob("23_Right*.csv"))[0]
                leg_file = sorted(activity_folder.glob("2C_Leg*.csv"))[0]

                left_df = read_sensor_file(left_file).add_prefix("Left_Wrist_")
                right_df = read_sensor_file(right_file).add_prefix("Right_Wrist_")
                leg_df = read_sensor_file(leg_file).add_prefix("Upper_Leg_")

                merged_trial = pd.concat([left_df, right_df, leg_df], axis=1)

                merged_trial["Activity"] = activity_label

                person_merged_rows.append(merged_trial)

                folders_processed += 1
                rows_added += len(merged_trial)

            except Exception as error:
                status = f"Error: {error}"

        summary_rows.append({
            "Person": person,
            "Activity": activity,
            "Folders Found": len(activity_folders),
            "Folders Processed": folders_processed,
            "Rows Added": rows_added,
            "Status": status
        })

    # Save one CSV per person after all activities are processed
    if person_merged_rows:
        person_final_df = pd.concat(person_merged_rows, ignore_index=True)

        person_out_file = merged_out_dir / f"{person}_Merged.csv"
        person_final_df.to_csv(person_out_file, index=False)

        print(f"Saved merged CSV for {person}: {person_out_file}")
    else:
        print(f"No merged data was created for {person}.")

display(pd.DataFrame(summary_rows))

Saved merged CSV for Hiruni: old_sensor_merged\Hiruni_Merged.csv
Saved merged CSV for Mineth: old_sensor_merged\Mineth_Merged.csv
Saved merged CSV for Tanushka: old_sensor_merged\Tanushka_Merged.csv
Saved merged CSV for Thinula: old_sensor_merged\Thinula_Merged.csv


,Person,Activity,Folders Found,Folders Processed,Rows Added,Status
0,Hiruni,Bending,4,4,2185,OK
1,Hiruni,Idle,1,1,2291,OK
2,Hiruni,Picking,1,1,2085,OK
3,Hiruni,Pushing,1,1,2127,OK
4,Mineth,Bending,3,3,894,OK
5,Mineth,Idle,1,1,1919,OK
6,Mineth,Picking,1,1,3481,OK
7,Mineth,Pushing,1,1,1947,OK
8,Tanushka,Bending,4,4,1017,OK
9,Tanushka,Idle,1,1,1780,OK


In [9]:
import pandas as pd
from pathlib import Path
from IPython.display import display

DATASET_PATH = Path("new_sensor_datasets")

people = ["Hiruni", "Mineth", "Tanushka", "Thinula", "Sineth"]

activities = ["Bending", "Idle", "Picking", "Pushing"]

activity_labels = {
    "Bending": 0,
    "Idle": 1,
    "Picking": 2,
    "Pushing": 3
}

merged_out_dir = Path("new_sensor_merged")
merged_out_dir.mkdir(parents=True, exist_ok=True)


def read_sensor_file(file_path: Path) -> pd.DataFrame:
    # New files are tab-separated .txt files
    df = pd.read_csv(file_path, sep="\t")

    required_columns = ["AccX(g)", "AccY(g)", "AccZ(g)"]

    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise KeyError(f"Missing columns in {file_path}: {missing_columns}")

    acc_df = df[required_columns].copy()

    acc_df.rename(columns={
        "AccX(g)": "Acc_X",
        "AccY(g)": "Acc_Y",
        "AccZ(g)": "Acc_Z"
    }, inplace=True)

    return acc_df


summary_rows = []

for person in people:
    person_merged_rows = []

    person_folder = DATASET_PATH / person

    if not person_folder.exists():
        summary_rows.append({
            "Person": person,
            "Activity": "ALL",
            "Folders Found": 0,
            "Folders Processed": 0,
            "Rows Added": 0,
            "Status": "Person folder not found"
        })
        continue

    for activity in activities:
        
        activity_label = activity_labels[activity]

        activity_folders = sorted([
            folder for folder in person_folder.glob(f"{activity}*")
            if folder.is_dir()
        ])

        folders_processed = 0
        rows_added = 0
        status = "OK"

        for activity_folder in activity_folders:            
            try:
                left_candidates = list(activity_folder.glob(f"Left_{activity}*.txt"))
                right_candidates = list(activity_folder.glob(f"Right_{activity}*.txt"))

                leg_candidates = [
                    file for file in activity_folder.glob("*.txt")
                    if "leg" in file.stem.lower()
                ]

                if not left_candidates or not right_candidates or not leg_candidates:
                    raise FileNotFoundError(
                        f"Missing sensor file in {activity_folder}. "
                        f"left={len(left_candidates)}, "
                        f"right={len(right_candidates)}, "
                        f"leg={len(leg_candidates)}, "
                        f"available={[file.name for file in activity_folder.glob('*.txt')]}"
                    )

                left_file = sorted(left_candidates)[0]
                right_file = sorted(right_candidates)[0]
                leg_file = sorted(leg_candidates)[0]

                left_df = read_sensor_file(left_file).add_prefix("Left_Wrist_")
                right_df = read_sensor_file(right_file).add_prefix("Right_Wrist_")
                leg_df = read_sensor_file(leg_file).add_prefix("Upper_Leg_")

                
                min_len = min(len(left_df), len(right_df), len(leg_df))

                left_df = left_df.iloc[:min_len].reset_index(drop=True)
                right_df = right_df.iloc[:min_len].reset_index(drop=True)
                leg_df = leg_df.iloc[:min_len].reset_index(drop=True)

                merged_trial = pd.concat([left_df, right_df, leg_df], axis=1)
                

                merged_trial["Activity"] = activity_label

                person_merged_rows.append(merged_trial)

                folders_processed += 1
                rows_added += len(merged_trial)

            except Exception as error:
                status = f"Error: {error}"

        summary_rows.append({
            "Person": person,
            "Activity": activity,
            "Folders Found": len(activity_folders),
            "Folders Processed": folders_processed,
            "Rows Added": rows_added,
            "Status": status
        })

    # Save one CSV per person after all activities are processed
    if person_merged_rows:
        person_final_df = pd.concat(person_merged_rows, ignore_index=True)

        person_out_file = merged_out_dir / f"{person}_Merged.csv"
        person_final_df.to_csv(person_out_file, index=False)

        print(f"Saved merged CSV for {person}: {person_out_file}")
    else:
        print(f"No merged data was created for {person}.")

display(pd.DataFrame(summary_rows))

Saved merged CSV for Hiruni: new_sensor_merged\Hiruni_Merged.csv
Saved merged CSV for Mineth: new_sensor_merged\Mineth_Merged.csv
Saved merged CSV for Tanushka: new_sensor_merged\Tanushka_Merged.csv
Saved merged CSV for Thinula: new_sensor_merged\Thinula_Merged.csv
Saved merged CSV for Sineth: new_sensor_merged\Sineth_Merged.csv


,Person,Activity,Folders Found,Folders Processed,Rows Added,Status
0,Hiruni,Bending,1,1,102,OK
1,Hiruni,Idle,1,1,600,OK
2,Hiruni,Picking,1,1,600,OK
3,Hiruni,Pushing,1,1,600,OK
4,Mineth,Bending,1,1,69,OK
5,Mineth,Idle,1,1,593,OK
6,Mineth,Picking,1,1,600,OK
7,Mineth,Pushing,1,1,603,OK
8,Tanushka,Bending,1,1,84,OK
9,Tanushka,Idle,1,1,605,OK
